In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Localizar y cargar credenciales del .env
posibles_rutas = [Path(".env"), Path("../Scripts/.env"), Path("Scripts/.env")]
env_path = next((p for p in posibles_rutas if p.exists()), None)
load_dotenv(dotenv_path=env_path)

storage_options = {
    "key": os.getenv("R2_ACCESS_KEY"),
    "secret": os.getenv("R2_SECRET_KEY"),
    "client_kwargs": {
        "endpoint_url": f"https://{os.getenv('R2_ACCOUNT_ID')}.eu.r2.cloudflarestorage.com",
        "region_name": "eeur"
    },
    "config_kwargs": {"s3": {"addressing_style": "path"}}
}

# Leer el parquet desde Cloudflare R2
bucket = os.getenv("R2_BUCKET_NAME")
ruta_r2 = f"s3://{bucket}/raw/yfinance/stocks_2010_2023.parquet"
df = pd.read_parquet(ruta_r2, engine="pyarrow", storage_options=storage_options)

print("=== 1. RESUMEN ESTRUCTURAL DEL ARCHIVO ===")
print(f"Dimensiones         : {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Fecha inicial       : {df.index.min().strftime('%Y-%m-%d')}")
print(f"Fecha final         : {df.index.max().strftime('%Y-%m-%d')}")
print(f"Fechas duplicadas   : {df.index.duplicated().sum()}")
print(f"Orden cronológico   : {df.index.is_monotonic_increasing}")
print(f"Total nulos (NaN)   : {df.isna().sum().sum()}")
print("\nTipos de datos por bloque:")
print(df.dtypes.value_counts())

print("\n=== 2. SESIONES BURSÁTILES POR AÑO ===")
sesiones_anio = df.groupby(df.index.year).size()
print(sesiones_anio.to_frame(name="Sesiones").T)

# Verificación de Nulos, Regla High >= Low y Rangos por Banco
Se certifica banco por banco si hay nulos, si se cumple que el máximo diario supera al mínimo, y en qué rango de precios y volumen se mueve cada entidad.

In [ ]:
tickers = ["JPM", "BAC", "WFC", "GS", "MS", "C"]
resumen_calidad = []

for t in tickers:
    cols_banco = [f"{metric}_{t}" for metric in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
    
    high = df[f"High_{t}"]
    low = df[f"Low_{t}"]
    open_p = df[f"Open_{t}"]
    close = df[f"Close_{t}"]
    adj_close = df[f"Adj Close_{t}"]
    vol = df[f"Volume_{t}"]
    
    resumen_calidad.append({
        "Banco": t,
        "Nulos Totales": df[cols_banco].isna().sum().sum(),
        "Fallos High < Low": (high < low).sum(),
        "Fallos High == Low": (high == low).sum(),
        "Fallos Open/Close fuera de [Low, High]": ((high < np.maximum(open_p, close) - 1e-4) | (low > np.minimum(open_p, close) + 1e-4)).sum(),
        "Volumen <= 0": (vol <= 0).sum(),
        "Close Mín ($)": close.min(),
        "Close Máx ($)": close.max(),
        "Adj Close Mín ($)": adj_close.min(),
        "Adj Close Máx ($)": adj_close.max()
    })

df_calidad = pd.DataFrame(resumen_calidad).set_index("Banco")
print("=== 3. AUDITORÍA DE NULOS, COHERENCIA HIGH/LOW Y RANGOS ===")
display(df_calidad.round(2))


Tabla y Gráfico de la Media Anual por Banco (2010–2023)
Calcula el precio medio por año de cada banco y genera una cuadrícula de 6 gráficos (2x3) comparando la media anual del precio nominal (Close) frente al ajustado (Adj Close).

In [ ]:
# 1. Calcular medias anuales de Close y Adj Close
cols_close = [f"Close_{t}" for t in tickers]
cols_adj = [f"Adj Close_{t}" for t in tickers]

medias_close = df[cols_close].groupby(df.index.year).mean()
medias_adj = df[cols_adj].groupby(df.index.year).mean()

print("=== 4. TABLA DE PRECIO MEDIO ANUAL (ADJ CLOSE en USD) ===")
medias_adj_clean = medias_adj.copy()
medias_adj_clean.columns = tickers
medias_adj_clean.index.name = "Año"
display(medias_adj_clean.round(2))

# 2. Graficar la evolución de la media anual por banco (Subplots 2x3)
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.flatten()

anios = medias_close.index

for i, t in enumerate(tickers):
    ax = axes[i]
    ax.plot(anios, medias_close[f"Close_{t}"], marker="o", linestyle="--", color="steelblue", label="Close (Nominal)", alpha=0.8)
    ax.plot(anios, medias_adj[f"Adj Close_{t}"], marker="s", linewidth=2, color="darkblue", label="Adj Close (Ajustado)")
    
    ax.set_title(f"{t} - Precio Medio Anual (2010-2023)", fontsize=12, fontweight="bold")
    ax.set_ylabel("USD ($)")
    ax.set_xticks(anios[::2])  # Mostrar años de 2 en 2 para que no se solapen
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper left", fontsize=9)

plt.suptitle("Evolución de la Media Anual por Banco: Nominal vs. Ajustado (2010 - 2023)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()